# Human Action Detection Pipeline
This notebook implements an end-to-end machine learning pipeline to classify physical activities using multi-sensor body measurements from the MHEALTH dataset.

# Step 1: Data Preprocessing & Class Balancing
In this step, we load the raw dataset (`mhealth_raw_data.csv`), handle class imbalance by downsampling activity `0` (null state) to 40,000 rows, split the dataset into features ($X$) and target ($y$), perform a 75/25 train-test split, and scale feature data using `RobustScaler`.

In [ ]:
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler

# 1. Verify / Load raw CSV dataset
csv_file = "mhealth_raw_data.csv"
if not os.path.exists(csv_file):
    from generate_data import create_mhealth_dataset
    create_mhealth_dataset(filepath=csv_file)

print(f"Loading dataset: '{csv_file}'...")
df = pd.read_csv(csv_file)
print(f"Initial dataset shape: {df.shape}")

# 2. Handle class imbalance: filter & sample activity == 0
df_null = df[df['activity'] == 0]
df_non_null = df[df['activity'] != 0]

n_sample = min(40000, len(df_null))
print(f"Activity 0 row count before sampling: {len(df_null)}")
print(f"Sampling {n_sample} rows from activity == 0...")
df_null_sampled = df_null.sample(n=n_sample, random_state=42)

df_balanced = pd.concat([df_null_sampled, df_non_null], axis=0).reset_index(drop=True)
print(f"Balanced dataset shape: {df_balanced.shape}")
print("Activity class distribution:")
print(df_balanced['activity'].value_counts().sort_index())

# 3. Define features X and target y
X = df_balanced.drop(columns=['activity', 'subject'])
y = df_balanced['activity']

# 4. Train-Test Split (75/25)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)
print(f"X_train shape: {X_train.shape}, X_test shape: {X_test.shape}")

# 5. Robust Scaling
scaler = RobustScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print("Data preprocessing completed successfully.")

# Step 2: Model Training & Benchmarking
Here we define our custom result evaluation function `result_summarizer(y_true, y_pred, model_name)` which calculates Accuracy, Macro Precision, Macro Recall, and Macro F1 Score, and displays a Seaborn Confusion Matrix heatmap inline.

We train and compare three distinct machine learning models:
1. **Logistic Regression** (max_iter=1000)
2. **K-Neighbors Classifier** (k=5)
3. **Decision Tree Classifier** (max_depth=14)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier

def result_summarizer(y_true, y_pred, model_name="Model"):
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average='macro', zero_division=0)
    rec = recall_score(y_true, y_pred, average='macro', zero_division=0)
    f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)

    print(f"\n==========================================")
    print(f" Evaluation Results: {model_name}")
    print(f"==========================================")
    print(f" Accuracy        : {acc:.4f} ({acc * 100:.2f}%)")
    print(f" Macro Precision : {prec:.4f}")
    print(f" Macro Recall    : {rec:.4f}")
    print(f" Macro F1 Score  : {f1:.4f}")
    print(f"==========================================\n")

    # Plot Confusion Matrix
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(9, 7))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=True)
    plt.title(f"Confusion Matrix - {model_name}", fontsize=14)
    plt.xlabel("Predicted Label", fontsize=12)
    plt.ylabel("True Label", fontsize=12)
    plt.tight_layout()
    plt.show()

    return {
        'model': model_name,
        'accuracy': acc,
        'precision': prec,
        'recall': rec,
        'f1': f1
    }

# Define models
models = [
    ("Logistic Regression", LogisticRegression(max_iter=1000, random_state=42)),
    ("K-Neighbors Classifier (k=5)", KNeighborsClassifier(n_neighbors=5, n_jobs=-1)),
    ("Decision Tree (max_depth=14)", DecisionTreeClassifier(max_depth=14, random_state=42))
]

results = []

# Train and evaluate models
for name, model in models:
    print(f"Training {name}...")
    model.fit(X_train_scaled, y_train)
    print(f"Predicting with {name}...")
    y_pred = model.predict(X_test_scaled)
    
    metrics = result_summarizer(y_test, y_pred, model_name=name)
    results.append(metrics)

# Print Summary Table
print("\n" + "#"*55)
print(" FINAL MODEL PERFORMANCE COMPARISON")
print("#"*55)
print(f"{'Model':<30} | {'Accuracy':<10} | {'Macro F1':<10}")
print("-" * 55)
for r in results:
    print(f"{r['model']:<30} | {r['accuracy']*100:>8.2f}% | {r['f1']:>8.4f}")
print("#"*55 + "\n")

# Step 3: Best Model Export
In this step, we train the top-performing model (`KNeighborsClassifier` with $k=5$) on the training dataset, evaluate its overall score, and export the trained model to disk as `best_knn_model.pkl` using `joblib`.

In [ ]:
try:
    import joblib
    USE_JOBLIB = True
except ImportError:
    import pickle
    USE_JOBLIB = False

print("Training winning model: KNeighborsClassifier(n_neighbors=5)...")
best_knn = KNeighborsClassifier(n_neighbors=5, n_jobs=-1)
best_knn.fit(X_train_scaled, y_train)

train_acc = best_knn.score(X_train_scaled, y_train)
test_acc = best_knn.score(X_test_scaled, y_test)

print(f"Train Accuracy: {train_acc * 100:.2f}%")
print(f"Test Accuracy : {test_acc * 100:.2f}%")

model_filename = "best_knn_model.pkl"
print(f"Exporting model to '{model_filename}'...")
if USE_JOBLIB:
    joblib.dump(best_knn, model_filename)
    print("Model successfully exported using joblib.")
else:
    with open(model_filename, "wb") as f:
        pickle.dump(best_knn, f)
    print("Model successfully exported using pickle.")